# 02 — Cleaning & Wrangling

**Stage 2 of the Tier A pipeline.** The file passed nine of ten contract checks in `01`, so there is
little to *repair*. The work here is to (1) give columns analysis-safe names and types, (2) decide
what each column may be used for, which matters most for `CustomerID`, (3) build the derived columns
proposed in the data review (§7), and (4) write a cleaning log that says what changed and why.

In [1]:
import json, warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

ROOT = Path.cwd()
while not (ROOT / "archive" / "Mall_Customers.csv").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
ARCHIVE = ROOT / "archive"
RAW, PROC = ROOT / "data" / "raw", ROOT / "data" / "processed"
REPORTS, FIGS, MODELS = ROOT / "reports", ROOT / "reports" / "figures", ROOT / "models"
for p in (RAW, PROC, REPORTS, FIGS, MODELS):
    p.mkdir(parents=True, exist_ok=True)

KF_PATH = REPORTS / "_key_figures.json"


def _plain(v):
    if isinstance(v, dict):
        return {str(k): _plain(x) for k, x in v.items()}
    if hasattr(v, "item"):
        return v.item()
    if isinstance(v, (list, tuple)):
        return [_plain(x) for x in v]
    return v


def kf_update(**kw):
    """Append to reports/_key_figures.json — the only place the report reads numbers from."""
    kf = json.loads(KF_PATH.read_text()) if KF_PATH.exists() else {}
    kf.update({k: _plain(v) for k, v in kw.items()})
    KF_PATH.write_text(json.dumps(kf, indent=1, sort_keys=True))
    print(f"  _key_figures.json <- {', '.join(kw)}  ({len(kf)} figures total)")

raw = pd.read_csv(RAW / "mall_customers_raw_immutable.csv")
log = []


def note(step, rows_affected, why):
    log.append({"step": step, "rows_affected": rows_affected, "why": why})
    print(f"[{step}] {rows_affected} rows — {why}")


print(f"{len(raw)} rows loaded from the immutable copy")

200 rows loaded from the immutable copy


### 2.1 Names, types and roles

In [2]:
df = raw.rename(columns={"CustomerID": "customer_id", "Gender": "gender", "Age": "age",
                         "Annual Income (k$)": "income_k", "Spending Score (1-100)": "score"})
df["gender"] = df["gender"].str.strip().astype("category")
for c in ["customer_id", "age", "income_k", "score"]:
    df[c] = df[c].astype("int64")
note("rename + cast", len(df), "snake_case names; gender → category; numerics → int64")

ROLES = pd.DataFrame([
    ("customer_id", "key only", "encodes income rank (Spearman 1.00, `01` §1.5): never a feature"),
    ("gender", "descriptor", "profile and fairness check; not a clustering input"),
    ("age", "descriptor", "profile, driver analysis; tested as a clustering input in 05"),
    ("income_k", "clustering input", "annual income, k$; meaning (individual / household) undocumented"),
    ("score", "clustering input", "mall-assigned 1–100; not an outcome"),
], columns=["column", "role", "why"])
ROLES

[rename + cast] 200 rows — snake_case names; gender → category; numerics → int64


,column,role,why
0,customer_id,key only,"encodes income rank (Spearman 1.00, `01` §1.5)..."
1,gender,descriptor,profile and fairness check; not a clustering i...
2,age,descriptor,"profile, driver analysis; tested as a clusteri..."
3,income_k,clustering input,"annual income, k$; meaning (individual / house..."
4,score,clustering input,mall-assigned 1–100; not an outcome


### 2.2 Missing values, duplicates, rule checks

`01` found no nulls and no duplicate members, so the strategy per column is "none required". It is
still asserted here, so that a changed source file fails loudly instead of flowing through.

In [3]:
assert df.notna().all().all(), "nulls appeared — the source file changed"
assert not df.drop(columns="customer_id").duplicated().any(), "duplicate members appeared"
assert df["gender"].isin(["Female", "Male"]).all()
assert df["age"].between(18, 100).all() and df["score"].between(1, 100).all() and (df["income_k"] > 0).all()
note("null / duplicate / rule assertions", 0, "all pass; nothing imputed or dropped")

[null / duplicate / rule assertions] 0 rows — all pass; nothing imputed or dropped


### 2.3 Outliers — check, do not remove

The review flags a thin income tail (max 137k). Tukey's fences mark the extreme rows; they are
plausible incomes, so they stay. KMeans runs on standardised values, and `05` checks that the
segments do not depend on these points.

In [4]:
q1, q3 = df.income_k.quantile([0.25, 0.75])
upper = q3 + 1.5 * (q3 - q1)
df["income_outlier"] = df.income_k > upper
print(f"income Tukey upper fence: {upper:.1f}k  →  {df.income_outlier.sum()} rows above it: "
      f"{sorted(df.loc[df.income_outlier, 'income_k'].tolist())}")
for c in ["age", "score"]:
    lo, hi = df[c].quantile([0.25, 0.75])
    print(f"{c:6s} rows outside Tukey fences: {((df[c] < lo - 1.5*(hi-lo)) | (df[c] > hi + 1.5*(hi-lo))).sum()}")
note("income outlier flag", int(df.income_outlier.sum()), f"income > {upper:.1f}k flagged, kept (plausible)")

income Tukey upper fence: 132.8k  →  2 rows above it: [137, 137]
age    rows outside Tukey fences: 0
score  rows outside Tukey fences: 0
[income outlier flag] 2 rows — income > 132.8k flagged, kept (plausible)


### 2.4 Forensic columns from `01` — kept, never used as features

In [5]:
df["pair_id"] = (df.customer_id - 1) // 2 + 1
df["id_odd"] = df.customer_id % 2 == 1
assert df.groupby("pair_id").income_k.nunique().eq(1).all()
note("pair_id, id_odd", len(df), "construction signature from 01 §1.5 — forensic only, excluded from features")

[pair_id, id_odd] 200 rows — construction signature from 01 §1.5 — forensic only, excluded from features


### 2.5 Derived columns (data review §7)

Built from existing columns only. Each one's use is stated, and two carry a warning.

In [6]:
df["age_group"] = pd.cut(df.age, [17, 25, 35, 45, 55, 100],
                         labels=["18–25", "26–35", "36–45", "46–55", "56+"])
df["life_stage"] = pd.cut(df.age, [17, 29, 44, 59, 100],
                          labels=["Young adult", "Established", "Pre-retiree", "Senior"])
df["income_band"] = pd.cut(df.income_k, [0, 40, 69, 1000], labels=["≤ 40k", "41–69k", "≥ 70k"])
df["score_band"] = pd.cut(df.score, [0, 40, 59, 100], labels=["≤ 40", "41–59", "≥ 60"])
df["gender_female"] = (df.gender == "Female").astype(int)
df["age_over_40"] = df.age > 40

# Percentile-based gap: positive = spends above what income rank alone would suggest.
df["income_pct"] = df.income_k.rank(pct=True)
df["score_pct"] = df.score.rank(pct=True)
df["spending_gap"] = df.score_pct - df.income_pct
# Ratio kept because the review proposes it, but the score is not currency — compare only within a band.
df["score_per_income"] = df.score / df.income_k

med_i, med_s = df.income_k.median(), df.score.median()
df["income_quadrant"] = np.select(
    [(df.income_k >= med_i) & (df.score >= med_s), (df.income_k >= med_i) & (df.score < med_s),
     (df.income_k < med_i) & (df.score >= med_s)],
    ["high inc · high score", "high inc · low score", "low inc · high score"], "low inc · low score")
df["income_z"] = (df.income_k - df.income_k.mean()) / df.income_k.std(ddof=0)
df["score_z"] = (df.score - df.score.mean()) / df.score.std(ddof=0)

note("derived columns", len(df), "age_group, life_stage, income_band, score_band, gender_female, age_over_40, "
     "spending_gap, score_per_income, income_quadrant, income_z, score_z")
print()
print("income bands :", df.income_band.value_counts(sort=False).to_dict())
print("score bands  :", df.score_band.value_counts(sort=False).to_dict())
print("quadrants    :", df.income_quadrant.value_counts().to_dict())

[derived columns] 200 rows — age_group, life_stage, income_band, score_band, gender_female, age_over_40, spending_gap, score_per_income, income_quadrant, income_z, score_z

income bands : {'≤ 40k': 50, '41–69k': 74, '≥ 70k': 76}
score bands  : {'≤ 40': 63, '41–59': 72, '≥ 60': 65}
quadrants    : {'low inc · high score': 52, 'high inc · low score': 50, 'high inc · high score': 50, 'low inc · low score': 48}


| Column | Use | Caution |
|---|---|---|
| `income_band` / `score_band` | The 40 / 70 income and 40 / 60 score cut-points are the candidate *rule* thresholds tested in `05` | Fixed before clustering, so `05`'s agreement check is not circular |
| `spending_gap` | Over- / under-spenders relative to income rank | A rank difference, not money |
| `score_per_income` | Review §7 proposal | **The score is not currency**: the ratio rewards low income mechanically. Not used in any finding |
| `income_quadrant` | Median split, the simplest possible segmentation | Used as a baseline the clustering must beat |
| `income_z`, `score_z` | KMeans input | Population SD (ddof = 0), identical to `StandardScaler` |

### 2.6 Save, and write the cleaning log

In [7]:
assert np.allclose(df[["income_z", "score_z"]].mean(), 0) and np.allclose(df[["income_z", "score_z"]].std(ddof=0), 1)
df.to_parquet(PROC / "mall_clean.parquet", index=False)
pd.DataFrame(log).to_csv(PROC / "_cleaning_log.csv", index=False)
print(f"saved data/processed/mall_clean.parquet  ({df.shape[0]} rows × {df.shape[1]} cols)")
print(f"saved data/processed/_cleaning_log.csv   ({len(log)} steps)")

kf_update(n_rows_clean=len(df), n_rows_dropped=len(raw) - len(df), n_income_outliers=int(df.income_outlier.sum()),
          income_upper_fence=float(upper), n_derived_cols=11)
pd.DataFrame(log)

saved data/processed/mall_clean.parquet  (200 rows × 21 cols)
saved data/processed/_cleaning_log.csv   (5 steps)
  _key_figures.json <- n_rows_clean, n_rows_dropped, n_income_outliers, income_upper_fence, n_derived_cols  (33 figures total)


,step,rows_affected,why
0,rename + cast,200,snake_case names; gender → category; numerics ...
1,null / duplicate / rule assertions,0,all pass; nothing imputed or dropped
2,income outlier flag,2,"income > 132.8k flagged, kept (plausible)"
3,"pair_id, id_odd",200,construction signature from 01 §1.5 — forensic...
4,derived columns,200,"age_group, life_stage, income_band, score_band..."


## Stage 2 — Gate Checklist

- [x] Missing-value strategy documented per column: none required, and asserted so a changed file fails
- [x] Duplicates assessed: none (excluding the ID)
- [x] All columns have correct dtypes
- [x] Business-rule validations pass
- [x] Cleaning log exists (`data/processed/_cleaning_log.csv`): what changed, rows affected, why
- [x] Cleaned data saved to `data/processed/mall_clean.parquet`

**Iteration log.** Pass 1 treated `CustomerID` as a plain key. The construction audit in `01` showed it
encodes income rank and pair membership, so pass 2 split it into a key (`customer_id`) and forensic flags
(`pair_id`, `id_odd`) that are barred from every feature set.